# 06 · Apuração parcial na ordem real do TSE

Parte 18. Pergunta: com x % dos votos válidos já recebidos, na **ordem real em que o TSE recebeu os boletins de urna (BU)**, o modelo `bb_slx` ajustado nos municípios já apurados prevê o resultado final melhor que a contagem bruta e que o swing uniforme? Regras e resultado em [doc 11](../docs/pressupostos/11-apuracao-parcial.md).

Fonte: `data/interim/tse_bweb.parquet` (seção a seção, 2022 e 2026; coletor `src/coleta/tse_bweb.py`). Funções em `src/apuracao.py`. Tabelas em `results/tables/apuracao_*.csv`, figuras em `results/figures/apuracao_*.html`.

**Regra (fixada antes de ver o resultado, não muda).** Cortes de 5, 10, 25, 50, 67 e 90 % dos votos válidos recebidos no Brasil (doméstico), em 2022 e em 2026. O modelo é **útil** se o erro absoluto do Brasil **e** o MAE regional (5 regiões) forem menores que os da (i) contagem bruta e da (ii) swing uniforme em **pelo menos 4 dos 6 cortes, em cada ano**. Alvo: o resultado final real.

**Desenho.**
- Por município, (y_c, n_c) = votos de Lula e válidos das seções recebidas até o corte; n_u = aptos das seções ainda não recebidas × (válidos ÷ aptos do 1º turno do mesmo ano no município).
- Modelo: `bb_slx` (a especificação final, via `Candidato.ajustar`) ajustado em (y_c, n_c) dos municípios com n_c > 0; prevê π̂ do restante. 2026 usa `MAPA_2026_PRE`; 2022 usa `MAPA_FIT_2022` (covariáveis anteriores a 2022). Padronização só nas linhas de treino. Total = Σ y_c + Σ π̂·n_u, por região e Brasil.
- (i) contagem bruta: share já apurado. (ii) swing uniforme: swing nacional médio dos apurados contra o 2º turno anterior (2018 para 2022; 2022 para 2026), somado ao share anterior de cada município não apurado.
- Boa Esperança do Norte (MT, código TSE 73709) não está no parquet do modelo: seus votos apurados entram nos três métodos e no alvo; o restante usa o share do método na UF.
- Intervalos de 95 %: `simular_votos_grupo` (β ~ N(β̂, Σ Conley 200 km), choque de UF com σ_UF, beta-binomial) sobre o restante, somado ao já apurado. σ_UF = 0,0564 (Parte 16 do notebook 05) e corte de Conley de 200 km (Parte 15) nos dois anos.

In [1]:
import sys, json, warnings, time
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
warnings.filterwarnings("always", message="Covariância não PSD")
from statsmodels.tools.sm_exceptions import PerfectSeparationError
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import especificacoes as E, espacial as S, backtest as BT, apuracao as A, graficos as G, pressupostos as PR, dados as D
from src.especificacoes import Candidato, Termos
pd.options.display.float_format = "{:,.4f}".format
T0 = time.time()
TAB, FIG = ROOT / "results/tables", ROOT / "results/figures"
for p in (TAB, FIG): p.mkdir(parents=True, exist_ok=True)
SEMENTE, B = 7, 1000   # corte e σ_UF vêm de results/tables (Partes 15 e 16 do notebook 05)
ANOS = (2022, 2026)
BD = D.carregar_base(ROOT)   # municípios, geometria na mesma ordem e pesos Queen (compartilhado com o notebook 05)
dfm, W = BD.df, BD.W
lat, lon = dfm["lat"].to_numpy(float), dfm["lon"].to_numpy(float)
cd_modelo = dfm["id_municipio_tse"].astype("int64")
dec = json.loads((TAB / "final_decisao.json").read_text())
tf = dec["termos_final"]
CORTE_KM = float(dec["corte_final"])
SIGMA_UF = float(pd.read_csv(TAB / "final_cobertura.csv").query("covariancia == 'conley' and sigma == 'sigma_uf'").valor_sigma.iloc[0])
assert dec["modelo_final"] == "bb_slx"
CF = Candidato("bb_slx", "betabin", "logit", Termos(vars=tuple(v for v in tf["vars"] if not v.startswith("w_")),
                                                      slx=tuple(tf["slx"]), disp=tf["disp"],
                                                      regiao_fundir=tuple(tuple(g) for g in tf["regiao_fundir"])))
secoes = pd.read_parquet(ROOT / "data/interim/tse_bweb.parquet")
print(len(dfm), "municípios no modelo;", secoes.groupby("ano").size().to_dict(), "seções;", "termos:", len(CF.termos.vars), "| disp:", CF.termos.disp)

5570 municípios no modelo; {2022: 471010, 2026: 497897} seções; termos: 24 | disp: regiao_logn


## Taxa de válidos do 1º turno como estimador de n_u

`n_u` precisa de uma taxa válidos ÷ aptos anterior ao 2º turno. A leitura literal (só Lula + adversário no 1º turno) ignora o voto de quem escolheu um terceiro candidato no 1º turno e vota nos dois no 2º; a alternativa é o total de válidos do 1º turno. A tabela compara as duas contra o n real do 2º turno (informação só para escolher o estimador, não entra nos métodos).

In [2]:
L = []
for ano in ANOS:
    aptos1 = dfm[f"aptos_1t_{ano}"].astype(float)
    real_r = dfm[f"n_2t_{ano}"] / aptos1
    for nome, num in (("dois_candidatos_1t", dfm[f"votos_pt_1t_{ano}"] + dfm[f"votos_bolsonaro_1t_{ano}"]),
                      ("todos_validos_1t", dfm[f"validos_1t_{ano}"])):
        r = num.astype(float) / aptos1
        L.append({"ano": ano, "taxa": nome, "soma_estimada_sobre_real": float((r * aptos1).sum() / (real_r * aptos1).sum()),
                  "mape_municipal_pct": float((abs(r - real_r) / real_r).mean() * 100)})
TAXAS = pd.DataFrame(L); TAXAS.to_csv(TAB / "apuracao_taxa_nu.csv", index=False)
print(TAXAS.to_string(index=False))

 ano               taxa  soma_estimada_sobre_real  mape_municipal_pct
2022 dois_candidatos_1t                    0.9139              6.5568
2022   todos_validos_1t                    0.9973              2.1001
2026 dois_candidatos_1t                    0.8866              9.4714
2026   todos_validos_1t                    0.9617              4.0954


## Ordem real e cortes

Os cortes seguem a curva acumulada por instante de recebimento do BU (horário de Brasília). A virada de 2022 (Lula passa de Bolsonaro) está validada no coletor em 18:44 a 68,1 % recebido. O % efetivo de cada corte é um pouco maior que o nominal, porque seções do mesmo instante entram juntas.

In [3]:
L = []
for ano in ANOS:
    s = secoes[secoes.ano == ano]
    for pct in A.CORTES:
        rec = A.recebidas_ate(s, pct)
        L.append({"ano": ano, "corte": pct, "instante": A.instante_do_corte(s, pct), "pct_efetivo": A.pct_efetivo(s, rec),
                  "secoes_recebidas": int(rec.sum()), "municipios_com_voto": int((s[rec].groupby("cd_municipio_tse")["validos"].sum() > 0).sum()),
                  "ufs": int(s[rec]["uf"].nunique())})
CORTES = pd.DataFrame(L); print(CORTES.to_string(index=False))

 ano  corte                  instante  pct_efetivo  secoes_recebidas  municipios_com_voto  ufs
2022      5 2022-10-30 17:25:04-03:00       5.0026             23247                 1412   27
2022     10 2022-10-30 17:33:30-03:00      10.0189             47366                 2127   27
2022     25 2022-10-30 17:55:38-03:00      25.0139            119098                 3147   27
2022     50 2022-10-30 18:26:40-03:00      50.0282            237154                 4008   27
2022     67 2022-10-30 18:42:53-03:00      67.0124            316393                 4546   27
2022     90 2022-10-30 19:08:37-03:00      90.0097            423267                 5328   27
2026      5 2026-10-25 19:28:16-03:00       5.0065             24819                 1497   26
2026     10 2026-10-25 19:38:08-03:00      10.0063             49487                 2111   27
2026     25 2026-10-25 20:02:44-03:00      25.0042            125171                 2964   27
2026     50 2026-10-25 20:26:34-03:00      50.0104

## Ajuste, estimativas e intervalos por corte

Os intervalos saem duas vezes: com n_u fixo e com n_u simulado (regra do [doc 11](../docs/pressupostos/11-apuracao-parcial.md), seção 3). Os 6 cortes são aninhados no tempo e as 6 unidades (5 regiões e Brasil) são correlacionadas: a tabela de cortes cobertos por unidade e ano é descritiva, não um teste de calibração, e não se soma entre unidades.

In [4]:
LIN, LIN_REG, LIN_IC, LIN_ICN, LIN_SD, LIN_REP, LIN_NU, FALHAS = [], [], [], [], [], [], [], []
PI_CORTE = {}
# covariáveis defasadas (SLX) de cada ano, construídas uma vez e reusadas pela sensibilidade a n_u
SLX = {ano: S.adicionar_slx(dfm, E.MAPA_2026 if ano == 2026 else BT.MAPA_FIT_2022, list(tf["slx"]), W) for ano in ANOS}
for ano in ANOS:
    s = secoes[secoes.ano == ano].reset_index(drop=True)
    df2, mapa = SLX[ano]
    prois = {k: v for k, v in mapa.items() if k in CF.termos.vars and f"2t_{ano}" in v}
    assert not prois, prois   # nenhuma covariável do modelo vem do 2º turno do próprio ano
    for pct in A.CORTES:
        rec = A.recebidas_ate(s, pct)
        base = A.montar_base(s, dfm, ano, rec)
        real = A.alvo(base)
        info = {"ano": ano, "corte": pct, "pct_efetivo": A.pct_efetivo(s, rec)}
        ests = {"bruta": A.estimativa_bruta(base), "swing": A.estimativa_swing(base)}
        LIN_NU.append({**info, "n_u_estimado": float(base.n_u.sum()), "restante_real": float((base.n_final - base.n_c).sum()),
                       "razao": float(base.n_u.sum() / (base.n_final - base.n_c).sum())})
        pi = None
        try:
            bm = base.reindex(cd_modelo)
            pm = A.ajustar_parcial(CF, df2, mapa, bm.y_c.to_numpy(), bm.n_c.to_numpy(), bm.n_u.to_numpy(), lat, lon, CORTE_KM)
            pi = pd.Series(pm.pi, index=bm.index)
            ests["modelo"] = A.estimativa_modelo(base, pi)
            ic = A.intervalos(pm, bm, base, A.share_uf_modelo(base, pi), CF.ligacao, SIGMA_UF, B, SEMENTE)
            ic = ic.assign(real=real.reindex(ic.index), estimativa=ests["modelo"].reindex(ic.index))
            ic["dentro"] = (ic.real >= ic.q025) & (ic.real <= ic.q975)
            for r_, x in ic.iterrows():
                LIN_IC.append({**info, "regiao": r_, "estimativa_pct": 100 * x.estimativa, "q025_pct": 100 * x.q025, "q975_pct": 100 * x.q975,
                               "real_pct": 100 * x.real, "largura_pp": 100 * (x.q975 - x.q025), "dentro_ic95": bool(x.dentro)})
            fk = A.fatores_nu(base, B, SEMENTE)   # n_u simulado: mesma semente e B, só seções já recebidas
            icn = A.intervalos(pm, bm, base, A.share_uf_modelo(base, pi), CF.ligacao, SIGMA_UF, B, SEMENTE, fatores=fk)
            icn = icn.assign(real=real.reindex(icn.index))
            for r_, x in icn.iterrows():
                LIN_ICN.append({**info, "regiao": r_, "q025_pct": 100 * x.q025, "q975_pct": 100 * x.q975, "real_pct": 100 * x.real,
                                "largura_pp": 100 * (x.q975 - x.q025), "dentro_ic95": bool(x.real >= x.q025 and x.real <= x.q975)})
            LIN_SD.append({**info, "sd_uf_nu": A.sd_uf_nu(base)})
        except np.linalg.LinAlgError as e:   # matriz singular no ajuste conta como derrota do modelo; qualquer outro erro é bug e interrompe
            FALHAS.append({**info, "erro": repr(e)[:200]})
        PI_CORTE[(ano, pct)] = pi
        for met, est in ests.items():
            eb, mr = A.erros(est, real)
            LIN.append({**info, "metodo": met, "estimativa_brasil_pct": 100 * est["Brasil"], "real_brasil_pct": 100 * real["Brasil"],
                        "erro_brasil_com_sinal_pp": 100 * (est["Brasil"] - real["Brasil"]), "erro_brasil_pp": eb, "mae_regional_pp": mr})
            for r_ in real.index.drop("Brasil"):
                LIN_REG.append({**info, "metodo": met, "regiao": r_, "estimativa_pct": 100 * est[r_], "real_pct": 100 * real[r_], "erro_pp": 100 * (est[r_] - real[r_])})
        rep = A.representatividade(base, pi)
        for _, x in rep.iterrows(): LIN_REP.append({**info, **x.to_dict()})
        print(ano, pct, "ok", round(time.time() - T0), "s")
TAB_ERROS = pd.DataFrame(LIN)

2022 5 ok 7 s


2022 10 ok 12 s


2022 25 ok 18 s


2022 50 ok 25 s


2022 67 ok 32 s


2022 90 ok 38 s


... (2 outputs left out)

2026 25 ok 54 s


2026 50 ok 60 s


2026 67 ok 66 s


2026 90 ok 72 s


In [5]:
IC = pd.DataFrame(LIN_IC)
ICN = pd.DataFrame(LIN_ICN); ICN.to_csv(TAB / "apuracao_ic_nu_simulado.csv", index=False)
pd.DataFrame(LIN_SD).to_csv(TAB / "apuracao_sd_uf_nu.csv", index=False)
# n_u fixo (IC antigo) contra n_u simulado: cortes cobertos de 6, por unidade (5 regiões + Brasil) e ano
CHAVE = ["ano", "corte", "regiao"]
J = IC[CHAVE + ["dentro_ic95", "largura_pp"]].merge(ICN[CHAVE + ["dentro_ic95", "largura_pp"]], on=CHAVE, suffixes=("_fixo", "_sim"))
J["delta_largura_pp"] = J.largura_pp_sim - J.largura_pp_fixo
COB_UNI = J.groupby(["regiao", "ano"]).agg(cortes=("corte", "size"), cobertos_n_u_fixo=("dentro_ic95_fixo", "sum"), cobertos_n_u_simulado=("dentro_ic95_sim", "sum"),
                                         mediana_delta_largura_pp=("delta_largura_pp", "median")).reset_index().rename(columns={"regiao": "unidade"})
COB_UNI.to_csv(TAB / "apuracao_cobertura_unidade.csv", index=False)
COBERTURA = IC.groupby(["ano", "corte"]).agg(dentro=("dentro_ic95", "sum"), total=("dentro_ic95", "size"), largura_brasil_pp=("largura_pp", "last")).reset_index()
COBERTURA["cobertura"] = COBERTURA.dentro / COBERTURA.total
cob = COBERTURA.set_index(["ano", "corte"])["cobertura"]
TAB_ERROS["cobertura_ic95"] = [cob.get((a, c), np.nan) if m == "modelo" else np.nan for a, c, m in zip(TAB_ERROS.ano, TAB_ERROS.corte, TAB_ERROS.metodo)]
for ano in ANOS:
    TAB_ERROS[TAB_ERROS.ano == ano].to_csv(TAB / f"apuracao_{ano}.csv", index=False)
pd.DataFrame(LIN_REG).to_csv(TAB / "apuracao_regioes.csv", index=False)
IC.to_csv(TAB / "apuracao_ic.csv", index=False)
REP = pd.DataFrame(LIN_REP); REP.to_csv(TAB / "apuracao_representatividade.csv", index=False)
NU = pd.DataFrame(LIN_NU); NU.to_csv(TAB / "apuracao_n_restante.csv", index=False)
print("ajustes que falharam:", FALHAS)
for ano in ANOS:
    print(f"\n== {ano}: erro absoluto do Brasil (p.p.) e MAE regional (p.p.)")
    t = TAB_ERROS[TAB_ERROS.ano == ano]
    print(t.pivot(index="corte", columns="metodo", values=["erro_brasil_pp", "mae_regional_pp"]).round(3).to_string())
print("\nCortes cobertos pelo IC 95 % (de 6), por unidade e ano: n_u fixo contra n_u simulado"); print(COB_UNI.to_string(index=False))
print("\nsd_UF de n_u por corte"); print(pd.DataFrame(LIN_SD).pivot(index="corte", columns="ano", values="sd_uf_nu").round(4).to_string())

ajustes que falharam: []

== 2022: erro absoluto do Brasil (p.p.) e MAE regional (p.p.)
       erro_brasil_pp               mae_regional_pp              
metodo          bruta modelo  swing           bruta modelo  swing
corte                                                            
5              4.5720 0.4420 0.6000          0.7740 0.9920 3.4500
10             3.0540 0.8010 1.5320          0.5190 0.8540 3.4270
25             2.2000 0.6740 1.6250          0.7430 0.5490 2.8820
50             1.2280 0.5100 1.0910          0.6650 0.4110 1.8080
67             0.9290 0.3150 0.6140          0.6020 0.2550 1.1590
90             0.3820 0.0880 0.1250          0.3070 0.0800 0.3440

== 2026: erro absoluto do Brasil (p.p.) e MAE regional (p.p.)
       erro_brasil_pp               mae_regional_pp              
metodo          bruta modelo  swing           bruta modelo  swing
corte                                                            
5              0.9650 0.9300 1.1930          1.7760 1.622

## A regra aplicada

In [6]:
REGRA = A.aplicar_regra(TAB_ERROS)
det = []
for (ano, c), p in TAB_ERROS.groupby(["ano", "corte"]):
    p = p.set_index("metodo")
    if "modelo" not in p.index:
        det.append({"ano": ano, "corte": c, "vence_brasil": False, "vence_regional": False, "vence": False}); continue
    vb = bool((p.loc["modelo", "erro_brasil_pp"] < p.drop(index="modelo")["erro_brasil_pp"]).all())
    vr = bool((p.loc["modelo", "mae_regional_pp"] < p.drop(index="modelo")["mae_regional_pp"]).all())
    det.append({"ano": ano, "corte": c, "vence_brasil": vb, "vence_regional": vr, "vence": vb and vr})
DET = pd.DataFrame(det)
REGRA["veredito"] = np.where(REGRA.util, "modelo útil", "modelo não útil")
DET.to_csv(TAB / "apuracao_regra_cortes.csv", index=False); REGRA.to_csv(TAB / "apuracao_regra.csv", index=False)
print(DET.to_string(index=False)); print(); print(REGRA.to_string(index=False))

 ano  corte  vence_brasil  vence_regional  vence
2022      5          True           False  False
2022     10          True           False  False
2022     25          True            True   True
2022     50          True            True   True
2022     67          True            True   True
2022     90          True            True   True
2026      5          True           False  False
2026     10          True            True   True
2026     25          True            True   True
2026     50          True            True   True
2026     67          True            True   True
2026     90          True            True   True

 ano  cortes  vitorias_modelo  minimo  util    veredito
2022       6                4       4  True modelo útil
2026       6                5       4  True modelo útil


In [7]:
# Sensibilidade a n_u: mesma regra, com cada definição da taxa. A adotada vem dos ajustes do laço principal; só as outras são refeitas
ADOTADA = (TAB_ERROS.merge(NU[["ano", "corte", "razao"]], on=["ano", "corte"]).rename(columns={"razao": "n_u_sobre_restante_real"})
           .assign(definicao=A.DEFINICAO_TAXA)[["definicao", "ano", "corte", "metodo", "erro_brasil_pp", "mae_regional_pp", "n_u_sobre_restante_real"]])
LS = []
for definicao in A.DEFINICOES_TAXA:
    if definicao == A.DEFINICAO_TAXA:
        LS.extend(ADOTADA.to_dict("records"))
        continue
    for ano in ANOS:
        s = secoes[secoes.ano == ano].reset_index(drop=True)
        df2, mapa = SLX[ano]
        for pct in A.CORTES:
            rec = A.recebidas_ate(s, pct)
            base = A.montar_base(s, dfm, ano, rec, definicao)
            real = A.alvo(base)
            ests = {"bruta": A.estimativa_bruta(base), "swing": A.estimativa_swing(base)}
            try:
                bm = base.reindex(cd_modelo)
                pm = A.ajustar_parcial(CF, df2, mapa, bm.y_c.to_numpy(), bm.n_c.to_numpy(), bm.n_u.to_numpy(), lat, lon, CORTE_KM)
                ests["modelo"] = A.estimativa_modelo(base, pd.Series(pm.pi, index=bm.index))
            except (np.linalg.LinAlgError, PerfectSeparationError, ValueError, FloatingPointError) as e:
                print("falhou", definicao, ano, pct, repr(e)[:120])
            restante = float((base.n_final - base.n_c).sum())
            for met, est in ests.items():
                eb, mr = A.erros(est, real)
                LS.append({"definicao": definicao, "ano": ano, "corte": pct, "metodo": met, "erro_brasil_pp": eb, "mae_regional_pp": mr,
                           "n_u_sobre_restante_real": float(base.n_u.sum()) / restante})
SENS = pd.DataFrame(LS)
REGRA_S = pd.concat([A.aplicar_regra(g).assign(definicao=d) for d, g in SENS.groupby("definicao")], ignore_index=True)
REGRA_S["veredito"] = np.where(REGRA_S.util, "modelo útil", "modelo não útil")
SENS.to_csv(TAB / "apuracao_sensibilidade_nu.csv", index=False)
print("Erro da taxa por município e razão da soma (tabela da seção da taxa):"); print(TAXAS.to_string(index=False))
print("\nn_u estimado sobre restante real, por corte"); print(SENS[SENS.metodo == "modelo"].pivot_table(index=["ano", "corte"], columns="definicao", values="n_u_sobre_restante_real").round(4).to_string())
print("\nErro do Brasil (p.p.) e MAE regional (p.p.) do modelo, por definição")
print(SENS[SENS.metodo.isin(["modelo", "swing"])].pivot_table(index=["ano", "corte"], columns=["metodo", "definicao"], values=["erro_brasil_pp", "mae_regional_pp"]).round(3).to_string())
print("\nVeredito pela regra, por definição"); print(REGRA_S.to_string(index=False))


Erro da taxa por município e razão da soma (tabela da seção da taxa):
 ano               taxa  soma_estimada_sobre_real  mape_municipal_pct
2022 dois_candidatos_1t                    0.9139              6.5568
2022   todos_validos_1t                    0.9973              2.1001
2026 dois_candidatos_1t                    0.8866              9.4714
2026   todos_validos_1t                    0.9617              4.0954

n_u estimado sobre restante real, por corte
definicao   dois_candidatos  todos_validos
ano  corte                                
2022 5               0.9140         0.9970
     10              0.9134         0.9967
     25              0.9124         0.9965
     50              0.9112         0.9962
     67              0.9112         0.9961
     90              0.9106         0.9955
2026 5               0.8859         0.9609
     10              0.8852         0.9602
     25              0.8825         0.9582
     50              0.8791         0.9557
     67            

In [8]:
fig = G.erro_por_corte(TAB_ERROS, ANOS, "erro_brasil_pp", "erro absoluto do Brasil (p.p.)", "Erro do Brasil por corte e método (escala log)")
G.salvar(fig, FIG / "apuracao_erro_brasil.html"); fig.show()
fig = G.erro_por_corte(TAB_ERROS, ANOS, "mae_regional_pp", "MAE regional (p.p.)", "MAE regional por corte e método (escala log)")
G.salvar(fig, FIG / "apuracao_mae_regional.html"); fig.show()

## Pressuposto declarado: as seções apuradas representam as não apuradas?

Dentro de um município, o modelo supõe que o restante tem o mesmo share que o apurado, dado o que as covariáveis dizem do município. Isso falha onde a ordem de chegada se correlaciona com o voto dentro do município (capitais grandes: zonas centrais e periferias chegam em tempos diferentes). Medida: nos 20 maiores municípios (por válidos finais) e nos demais, a diferença entre o share apurado (ou o π̂ do modelo) e o share **real do restante**, média ponderada pelos votos restantes reais, só em municípios com apuração iniciada e restante > 0.

In [9]:
REP_V = REP.copy()
print(REP_V.pivot_table(index=["ano", "corte"], columns="grupo", values=["pct_restante_real", "lacuna_apurado_pp", "abs_apurado_pp", "lacuna_modelo_pp", "abs_modelo_pp"]).round(2).to_string())

           abs_apurado_pp         abs_modelo_pp         lacuna_apurado_pp         lacuna_modelo_pp         pct_restante_real        
grupo              demais maiores        demais maiores            demais maiores           demais maiores            demais maiores
ano  corte                                                                                                                          
2022 5             5.8100  2.1300        2.7900  1.2700            1.1900  0.4200           0.2200 -0.7500           54.9500 45.0500
     10            4.7900  1.5100        2.5400  1.5600           -0.3900  1.2500          -0.6100 -0.9500           63.7900 36.2100
     25            3.8300  1.2700        2.4100  1.1200           -0.6100  0.2500          -1.1000 -0.6800           71.0300 28.9700
     50            3.3200  0.7300        2.4600  1.0900           -1.3500 -0.3400          -1.3500 -0.6700           72.2100 27.7900
     67            3.2300  1.0200        2.4200  1.1300           -1.

## Leitura

**Veredito pela regra fixada de antemão (modelo menor que a contagem bruta e que o swing uniforme, no erro do Brasil e no MAE regional, em pelo menos 4 de 6 cortes): modelo útil nos dois anos.**

- **2022:** vitórias em 4 de 6 cortes (25, 50, 67 e 90%). Perde o MAE regional a 5% (0,99 contra 0,77 da bruta) e a 10% (0,85 contra 0,52). Erro do Brasil: 0,44 / 0,80 / 0,67 / 0,51 / 0,32 / 0,09 p.p. nos cortes 5 a 90, contra 4,57 a 0,38 da bruta e 0,60 a 1,63 do swing.
- **2026:** vitórias em 5 de 6 cortes. Só perde o MAE regional a 5% (1,62 contra 1,49 do swing). Erro do Brasil: 0,93 / 0,06 / 0,25 / 0,17 / 0,08 / 0,02 p.p., contra bruta de 0,22 a 2,37 e swing de 0,14 a 1,19.
- O modelo vence no erro do Brasil nos 12 cortes dos dois anos. A margem de 2022 sobre a regra é mínima (4 de 6): a conclusão de 2022 é frágil, a de 2026 é mais folgada.
- **Cobertura do IC 95% por unidade e ano** (cortes cobertos de 6, tabela `apuracao_cobertura_unidade.csv`): o Brasil fica dentro em 6 de 6 nos dois anos; Nordeste 2022 (3) e Norte (4 em 2022, 3 em 2026) são as unidades mais abaixo do nominal. Simular a incerteza de n_u não mudou nenhuma contagem e alterou a largura em menos de 0,05 p.p. Os cortes são aninhados no tempo e as unidades são correlacionadas: a tabela é descritiva, não um teste de calibração. σ_UF de 2026 e corte de Conley foram reusados em 2022 (limitação).
- **Taxa de n_u:** a leitura literal (só os dois candidatos do 1º turno) erra 6,6% (2022) e 9,5% (2026) por município; a de todos os válidos erra 2,1% e 4,1% e foi a adotada. A escolha usou a comparação com o n final, e isso é declarado. Mesmo assim, n_u subestima o restante em 0,3% (2022) a 4,8% (2026).
- **Representatividade nas capitais:** os 20 maiores municípios concentram 24% a 46% do restante real. O share apurado erra o share do restante em 0,7 a 4,4 p.p. nesse grupo (2026 a 10%: apurado 3,6 p.p. abaixo do restante) e em 2,8 a 6,9 p.p. nos demais. O modelo reduz a lacuna nos dois grupos na maioria dos cortes, mas a hipótese de que as seções apuradas representam as restantes dentro do município só é corrigida em parte pelas covariáveis.